plan: Given the importance attribution of each explanation function, aggregate over trials by dividing each pixel value within a trial by the norm of the entire trial(to make each trial have distance 1 from the origin).
This makes each trial equally important while preserving the sign of each attribution value

In [3]:
import pathlib
import random
import copy
import gc
import numpy as np
import torch
import torchvision
import quantus
import torchvision
from captum.attr import *
import matplotlib.pyplot as plt
from omegaconf import OmegaConf
import yaml
import argparse

from models.s4net import S4PatchedFinalNet,TrunkNet, HeadNet
from data.data_loader import load_eeg_data, get_sliding_window_data, create_dataloader
import os
import mne
from tqdm import tqdm
from scipy.signal import hilbert
from sklearn.preprocessing import MinMaxScaler
from scipy.io import loadmat
from tqdm import tqdm

from pycircstat.tests import *
import statsmodels.multivariate.multivariate_ols as mv_ols
import statsmodels.api as sm 
import pandas as pd

from scipy.signal import welch
from scipy.integrate import simpson
import pickle

In [2]:
CFG_YAML = """
wandb:
 key: f0c92a0059bf12e2647f0a1c22fdcd12555fa6df
model:
dataset:
 data_directory: /home/marco/Documents/GitHub/tms_eeg_decoding/data
 #file_name: subject_{:03d}_preprocessed_combined_py.fif
 file_name: subject_{:03d}_preprocessed_combined_py.fif
 exclude_timepoints: 100
 subject_index: 1
 test_subject_indices: [1,2,13,24,26,27,29,34,35,41, 42,43,45,46,47,48,52,55,56,57,60,62,67,69,72,73,79,80,86,88,92,102]
 #test_subject_indices: [2]
training:
 training_start_len: 100
 pretrain_epochs: 100
 pretrain_lr: 0.0001
 val_window_len: 1
 epochs_per_window: 10
 num_warmup_epochs: 5
 num_epochs: 800
 slide_step: 1
 num_warmup_epochs_per_window: 0
 lr: 0.005 #maybe change back to 0.0001
 nll_beta: 0.001
 num_warmup_epochs: 0
 batch_size: 50 #better to use 50
 random_seed: 42
 precision: bf16
 kde_lambda: 0.5
 finetune_entire_model: true # Set to true to finetune the entire model, false for transformer only
exp_name: S4_S4EEGNet_ema
"""


def load_config():
    cfg = OmegaConf.create(yaml.safe_load(CFG_YAML))
    cfg.exp_name = f"{cfg.exp_name}_subject_{cfg.dataset.subject_index}"
    return cfg

def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--update_conf", nargs="*", help="Updates to the configuration in the form of key=value pairs", default=[])
    parser.add_argument("-f", "--fff", help="A dummy argument to handle IPython's default argument", default="1")
    return parser.parse_args()

def update_config(cfg, cli_args):
    for update in cli_args.update_conf:
        key, value = update.split("=")
        try:
            value = eval(value)
        except:
            pass
        OmegaConf.update(cfg, key, value, force_add=True)
    cfg.exp_name = cfg.exp_name + "_" + "_".join(cli_args.update_conf)
    print(OmegaConf.to_yaml(cfg))
    return cfg


def save_config(cfg):
    os.makedirs("conf/sweeps", exist_ok=True)
    os.makedirs("exp/withinsubs", exist_ok=True)
    with open(f"conf/sweeps/withinsubs_{cfg.exp_name}.yaml", "w") as f:
        f.write(OmegaConf.to_yaml(cfg))


In [72]:
def load_explanations_gradshap(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability"
    
    with open(os.path.join(load_path, f"subject_{subject_index}_results.pkl"), 'rb') as f:
        data = pickle.load(f)
        gradshap = data['explanations']
    return gradshap

In [73]:
def load_explanations_saliency(subject_index=2):
    load_path = "/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/saliency_explanations"

    saliency = np.load(os.path.join(load_path, f"saliency_explanations_subject_{subject_index}.npy"), allow_pickle=True)
    return saliency



In [11]:
def load_ch_names(subject_index=2):
    cfg = load_config()
    cfg.dataset.subject_index = subject_index
    _, _, _, _, _, _, _, ch_names = load_eeg_data(cfg)
    return ch_names

In [55]:
def get_channel_importances(explanations, ch_names, take_abs=False):


    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial
    
    channel_importances = np.zeros(len(ch_names))

    for trial_idx, trial_normed in enumerate(explanations_normed):
        if take_abs:
            channel_importances += np.mean(np.abs(trial_normed), axis=1)
        else:
            channel_importances += np.mean(trial_normed, axis=1)
    
    channel_importances_dic = {ch_names[i]: channel_importances[i] for i in range(len(ch_names))}

    return channel_importances_dic

In [43]:
def create_index_groups(n_samples, subject_index, group_size=100):
    """
    Create index groups for a given subject based on uncertainty values.
    
    Parameters:
    -----------
    uncertainties : array-like
        The uncertainties array for the subject
    subject_index : int
        Index of the subject
    group_size : int, default=100
        Size of each group
    
    Returns:
    --------
    dict
        Dictionary with subject_index as key and array of boolean index groups as value
    """
    index_groups_all = {}
    index_groups_subject = []
    
    start = 0
    while start < n_samples:
        end = min(start + group_size, n_samples-20)
        
        index_group = np.zeros(n_samples, dtype=bool)
        index_group[start:end] = True
        
        if np.sum(index_group) > 1:
            index_groups_subject.append(index_group)
        
        start += group_size
    

    return index_groups_subject

In [64]:
index_groups = create_index_groups(explanations.shape[0], 2, group_size=100)

In [65]:
def get_channel_importances_time_indexed(explanations, ch_names, index_groups, take_abs=False):
    explanations_normed = np.zeros_like(explanations)
    for trial_idx, trial in enumerate(explanations):
        trial_normed = trial / np.linalg.norm(trial)
        explanations_normed[trial_idx] = trial_normed

    #channel_importances = np.zeros(( len(index_groups),len(ch_names)))
    channel_importances = []
    for i,index_group in enumerate(index_groups):
        explanations_index_group = explanations_normed[index_group]
        channel_importances_index_group = np.zeros(len(ch_names))
        
        for trial_idx, trial_normed in enumerate(explanations_index_group):
            if take_abs:
                # taking the mean within a trial is fine as different time poins are supposed to be more important than other
                channel_importances_index_group += np.mean(np.abs(trial_normed), axis=1)
            else:
                channel_importances_index_group += np.mean(trial_normed, axis=1)
    
        channel_importances_dic = {ch_names[i]: channel_importances_index_group[i] for i in range(len(ch_names))}
        channel_importances.append(channel_importances_dic)
    

    return channel_importances
    


In [90]:
ch_names = load_ch_names(2)
explanations_gradshap = load_explanations_gradshap(2)
channel_importances_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=False)
channel_importances_abs_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=True)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [63]:
def get_top_k_keys(d, k, reverse=True):

    # Sort the dictionary by values in descending order and get the top k keys
    top_k_keys = sorted(d, key=d.get, reverse=reverse)[:k]
    return top_k_keys

In [87]:
print(get_top_k_keys(channel_importances_gradshap, 10))
print(get_top_k_keys(channel_importances_gradshap, 10, reverse=False))
print(get_top_k_keys(channel_importances_abs_gradshap, 10))

['C4', 'PO8', 'CP4', 'P8', 'C6', 'F8', 'P2', 'P5', 'CP5', 'O2']
['FC2', 'F2', 'Cz', 'Iz', 'FC1', 'F1', 'Oz', 'AF7', 'FC4', 'TP8']
['PO8', 'C4', 'CP4', 'O2', 'P2', 'FC2', 'Iz', 'FC1', 'Cz', 'AF7']


In [93]:
channel_importances_gradshap_time_indexed = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=False)
channel_importances_gradshap_time_indexed_abs = get_channel_importances_time_indexed(explanations_gradshap, ch_names, index_groups, take_abs=True)

In [68]:
for index_group in channel_importances_time_indexed:
    print(get_top_k_keys(index_group, 10))

['PO8', 'P8', 'C4', 'CP5', 'CP4', 'F5', 'C5', 'FC3', 'P5', 'FT8']
['C4', 'PO8', 'CP4', 'P5', 'P8', 'P2', 'O1', 'F8', 'F5', 'C5']
['C4', 'PO8', 'CP4', 'P2', 'P8', 'C6', 'P5', 'F8', 'Fz', 'O1']
['C4', 'PO8', 'CP4', 'C6', 'F8', 'P8', 'O2', 'P2', 'P3', 'AF8']
['C4', 'PO8', 'CP4', 'C6', 'O2', 'F8', 'P2', 'TP7', 'P7', 'CP6']
['C4', 'CP4', 'PO8', 'C6', 'F8', 'O2', 'AF7', 'P3', 'AF8', 'F7']


In [70]:
for index_group in channel_importances_time_indexed:
    print(get_top_k_keys(index_group, 10, reverse=False))

['Iz', 'F2', 'C6', 'Oz', 'FC5', 'P1', 'FC6', 'CPz', 'Cz', 'F7']
['FC2', 'P7', 'F1', 'F2', 'Iz', 'Fpz', 'FC4', 'AF7', 'CP2', 'Cz']
['FC2', 'AF7', 'TP8', 'PO3', 'Iz', 'Cz', 'FC6', 'FC4', 'C1', 'Fp1']
['FC1', 'TP8', 'Oz', 'AF7', 'Pz', 'T8', 'FT7', 'C3', 'C1', 'Cz']
['FC2', 'FC1', 'C1', 'Cz', 'Fz', 'F2', 'F1', 'Fp2', 'AF4', 'CP1']
['F1', 'FC1', 'FC2', 'C1', 'F6', 'Fz', 'F2', 'AF3', 'Cz', 'F3']


In [71]:
for index_group in channel_importances_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

['FC1', 'Iz', 'CP4', 'P2', 'FC2', 'PO8', 'PO3', 'C4', 'F5', 'Cz']
['CP4', 'P2', 'PO8', 'C4', 'FC2', 'Iz', 'O2', 'FC1', 'PO3', 'P7']
['CP4', 'PO8', 'C4', 'P2', 'FC2', 'O2', 'Iz', 'FC1', 'AF7', 'C6']
['PO8', 'C4', 'CP4', 'O2', 'FC2', 'P2', 'Iz', 'FC1', 'C6', 'F2']
['C4', 'PO8', 'CP4', 'O2', 'FC2', 'C6', 'P2', 'Cz', 'Iz', 'CP6']
['C4', 'PO8', 'CP4', 'O2', 'FC2', 'CP6', 'C6', 'AF7', 'P2', 'Iz']


while most important channels (5 most important) seem to be somewhat robust, this claim can not be made for the lower channels,
where importance can quickly shift

## compare results when saliency is used as explanation function

In [91]:
ch_names = load_ch_names(2)
explanations_saliency = load_explanations_saliency(2)
channel_importances_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=False)
channel_importances_abs_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=True)

Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)


In [92]:
print(get_top_k_keys(channel_importances_gradshap, 10))
print(get_top_k_keys(channel_importances_saliency, 10))

print(get_top_k_keys(channel_importances_gradshap, 10, reverse=False))
print(get_top_k_keys(channel_importances_saliency, 10, reverse=False))

print(get_top_k_keys(channel_importances_abs_gradshap, 10))
print(get_top_k_keys(channel_importances_abs_saliency, 10))

['C4', 'PO8', 'CP4', 'P8', 'C6', 'F8', 'P2', 'P5', 'CP5', 'O2']
['O2', 'F7', 'Pz', 'FC5', 'CP3', 'P3', 'FC3', 'FC1', 'Fz', 'PO3']
['FC2', 'F2', 'Cz', 'Iz', 'FC1', 'F1', 'Oz', 'AF7', 'FC4', 'TP8']
['C6', 'AF8', 'CP6', 'P5', 'AF3', 'F8', 'AF4', 'FT8', 'P6', 'TP7']
['PO8', 'C4', 'CP4', 'O2', 'P2', 'FC2', 'Iz', 'FC1', 'Cz', 'AF7']
['PO8', 'C4', 'CP4', 'O2', 'P2', 'FC2', 'Iz', 'FC1', 'Cz', 'CPz']


In [94]:
channel_importances_saliency_time_indexed = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=False)
channel_importances_saliency_time_indexed_abs = get_channel_importances_time_indexed(explanations_saliency, ch_names, index_groups, take_abs=True)

In [96]:
for index_group in channel_importances_gradshap_time_indexed:
    print(get_top_k_keys(index_group, 10))

for index_group in channel_importances_saliency_time_indexed:
    print(get_top_k_keys(index_group, 10))

['PO8', 'P8', 'C4', 'CP5', 'CP4', 'F5', 'C5', 'FC3', 'P5', 'FT8']
['C4', 'PO8', 'CP4', 'P5', 'P8', 'P2', 'O1', 'F8', 'F5', 'C5']
['C4', 'PO8', 'CP4', 'P2', 'P8', 'C6', 'P5', 'F8', 'Fz', 'O1']
['C4', 'PO8', 'CP4', 'C6', 'F8', 'P8', 'O2', 'P2', 'P3', 'AF8']
['C4', 'PO8', 'CP4', 'C6', 'O2', 'F8', 'P2', 'TP7', 'P7', 'CP6']
['C4', 'CP4', 'PO8', 'C6', 'F8', 'O2', 'AF7', 'P3', 'AF8', 'F7']
['O2', 'Pz', 'F7', 'CP3', 'FC1', 'FC5', 'Fz', 'P3', 'C1', 'F1']
['O2', 'FC3', 'C1', 'F7', 'Pz', 'FC1', 'FC5', 'AF7', 'C3', 'CP3']
['F7', 'P3', 'O1', 'PO3', 'CP3', 'FC5', 'O2', 'FC3', 'Pz', 'Fz']
['P3', 'PO3', 'FC5', 'PO7', 'P7', 'O1', 'Fz', 'CP3', 'F1', 'FC3']
['FC5', 'C5', 'P3', 'FC3', 'CP3', 'CP5', 'PO3', 'Pz', 'FC1', 'F7']
['C5', 'P3', 'P5', 'CP5', 'T7', 'TP7', 'TP8', 'FC5', 'P7', 'PO3']


In [97]:
for index_group in channel_importances_gradshap_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

for index_group in channel_importances_saliency_time_indexed_abs:
    print(get_top_k_keys(index_group, 10))

['FC1', 'Iz', 'CP4', 'P2', 'FC2', 'PO8', 'PO3', 'C4', 'F5', 'Cz']
['CP4', 'P2', 'PO8', 'C4', 'FC2', 'Iz', 'O2', 'FC1', 'PO3', 'P7']
['CP4', 'PO8', 'C4', 'P2', 'FC2', 'O2', 'Iz', 'FC1', 'AF7', 'C6']
['PO8', 'C4', 'CP4', 'O2', 'FC2', 'P2', 'Iz', 'FC1', 'C6', 'F2']
['C4', 'PO8', 'CP4', 'O2', 'FC2', 'C6', 'P2', 'Cz', 'Iz', 'CP6']
['C4', 'PO8', 'CP4', 'O2', 'FC2', 'CP6', 'C6', 'AF7', 'P2', 'Iz']
['FC1', 'CP4', 'Iz', 'P2', 'CPz', 'PO8', 'FC2', 'PO3', 'Cz', 'C4']
['CP4', 'PO8', 'P2', 'C4', 'FC2', 'Iz', 'FC1', 'O2', 'PO3', 'CPz']
['PO8', 'C4', 'CP4', 'O2', 'P2', 'FC2', 'Iz', 'FC1', 'Cz', 'C6']
['PO8', 'C4', 'CP4', 'O2', 'FC2', 'C6', 'P2', 'Iz', 'Cz', 'FC1']
['C4', 'PO8', 'CP4', 'O2', 'C6', 'FC2', 'Cz', 'CP6', 'Iz', 'P2']
['PO8', 'C4', 'O2', 'CP4', 'C6', 'CP6', 'FC2', 'AF7', 'F2', 'Iz']


# store results for all subjects

In [98]:
all_subject_channel_importances_gradshap = {}
all_subject_channel_importances_gradshap_abs = {}
all_subject_channel_importances_saliency = {}
all_subject_channel_importances_saliency_abs = {}
cfg = load_config()
for subject_index in cfg.dataset.test_subject_indices:
    ch_names = load_ch_names(subject_index)
    explanations_gradshap = load_explanations_gradshap(subject_index)
    channel_importances_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=False)
    channel_importances_abs_gradshap = get_channel_importances(explanations_gradshap, ch_names, take_abs=True)

    explanations_saliency = load_explanations_saliency(subject_index)
    channel_importances_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=False)
    channel_importances_abs_saliency = get_channel_importances(explanations_saliency, ch_names, take_abs=True)
    
    all_subject_channel_importances_gradshap[subject_index] = channel_importances_gradshap
    all_subject_channel_importances_gradshap_abs[subject_index] = channel_importances_abs_gradshap
    all_subject_channel_importances_saliency[subject_index] = channel_importances_saliency
    all_subject_channel_importances_saliency_abs[subject_index] = channel_importances_abs_saliency

np.save("all_subject_channel_importances_gradshap.npy", all_subject_channel_importances_gradshap)
np.save("all_subject_channel_importances_gradshap_abs.npy", all_subject_channel_importances_gradshap_abs)
np.save("all_subject_channel_importances_saliency.npy", all_subject_channel_importances_saliency)
np.save("all_subject_channel_importances_saliency_abs.npy", all_subject_channel_importances_saliency_abs)


Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_001_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
510 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_002_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
721 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_013_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


...

Adding metadata with 1 columns
608 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_092_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_092_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
564 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
Loading EEG data...
Reading /home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif ...
    Found the data of interest:
        t =   -1005.00 ...    -805.20 ms
        0 CTF compensation matrices available


/home/marco/Documents/GitHub/tms_eeg_decoding/subject_interpretability/data/data_loader.py:38: RuntimeWarning: This filename (/home/marco/Documents/GitHub/tms_eeg_decoding/data/subject_102_preprocessed_combined_py.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(file_path) # (trials, channels, timepoints)


Adding metadata with 1 columns
789 matching events found
No baseline correction applied
0 projection items activated
(100, 60, 900)
(1, 60, 1)
